# 06 · Séparation train / val et entraînement de YOLOv8s

**Objectif.** Construire le jeu fusionné xView + DOTA à 22 classes, le séparer sans fuite entre tuiles d'une même image, entraîner YOLOv8s et l'évaluer classe par classe.

**Entrées.**
- tuiles xView et étiquettes DOTA déjà converties vers les 22 classes : dataset Kaggle `henocamavigan/aerial-unified-22classes` ;
- images DOTA v1 découpées : dataset Kaggle `kefanzhan/dotav1`.

**Sorties.**
- `WORK_DIR/train_data/` (liens symboliques), `data_merged.yaml` (copie : `configs/data_merged.yaml`) ;
- `split_sources.json` : liste des images sources de chaque côté (copie : `configs/split_sources.json`) ;
- run Ultralytics `xview_dota_unified_v1` : `results.csv`, `args.yaml`, `weights/best.pt` (copies : `results/run/`).

**Configuration.** `yolov8s.pt` pré-entraîné COCO, 150 époques prévues, `imgsz=640`, `batch=32`, `patience=25`, 2 × Tesla T4, Ultralytics 8.4.163, PyTorch 2.10.0, Python 3.12.13.
Résultats : `results/README.md`.

*Les cellules ont été remises dans l'ordre logique ; sur Kaggle elles avaient été exécutées dans un ordre différent au fil des reprises.
Les sorties texte d'origine (journal de reprise à l'époque 60, évaluation par classe) sont conservées. Quatre cellules de test SAHI, reprises dans le notebook 07, ont été retirées.*

In [ ]:
# --- Configuration des chemins ---
# Exécution d'origine sur Kaggle : KAGGLE_INPUT = /kaggle/input/datasets, WORK_DIR = /kaggle/working.
# En local, définir les variables d'environnement GOD_EYE_INPUT et GOD_EYE_WORK (ou modifier les valeurs ci-dessous).
import os
KAGGLE_INPUT = os.environ.get("GOD_EYE_INPUT", "/kaggle/input/datasets")
WORK_DIR = os.environ.get("GOD_EYE_WORK", "/kaggle/working")

# Dataset Kaggle qui reçoit les sauvegardes du run (à remplacer par le vôtre)
CKPT_DATASET_ID = "ayiamavigan/xview-dota-ckpt-backup"

## 1. Matériel

In [1]:
import torch
print("CUDA disponible:", torch.cuda.is_available())
print("Nombre de GPU:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

CUDA disponible: True
Nombre de GPU: 2
0 Tesla T4
1 Tesla T4


## 2. Séparation train / val par image source

Toutes les tuiles d'une même image xView vont du même côté : sinon des tuiles voisines (et des objets coupés en deux) se retrouveraient à la fois en train et en val, ce qui gonfle les métriques. 10 % des 846 images sources vont en validation (graine 42). DOTA garde sa séparation officielle train / val.

In [1]:
import os, random
from collections import defaultdict

random.seed(42)

xview_img_dir = f"{KAGGLE_INPUT}/henocamavigan/aerial-unified-22classes/xview_tiles/images"
xview_lbl_dir = f"{KAGGLE_INPUT}/henocamavigan/aerial-unified-22classes/xview_tiles/labels"

# Regrouper les tuiles par image source (stem = id avant le premier "_")
tiles_by_source = defaultdict(list)
for fname in os.listdir(xview_img_dir):
    source_id = fname.split("_")[0]
    tiles_by_source[source_id].append(fname)

source_ids = list(tiles_by_source.keys())
random.shuffle(source_ids)

n_val = max(1, int(len(source_ids) * 0.10))
val_sources = set(source_ids[:n_val])
train_sources = set(source_ids[n_val:])

print(f"Images sources xView: {len(source_ids)} | Val: {len(val_sources)} | Train: {len(train_sources)}")

Images sources xView: 846 | Val: 84 | Train: 762


In [2]:
BASE = f"{WORK_DIR}/train_data"
for sub in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(f"{BASE}/{sub}", exist_ok=True)

def link_folder(src_dir, dst_dir):
    for fname in os.listdir(src_dir):
        dst = f"{dst_dir}/{fname}"
        if not os.path.exists(dst):
            os.symlink(f"{src_dir}/{fname}", dst)

def link_xview_split(source_ids, split):
    for sid in source_ids:
        for fname in tiles_by_source[sid]:
            os.symlink(f"{xview_img_dir}/{fname}", f"{BASE}/images/{split}/{fname}")
            lbl_name = fname.replace(".jpg", ".txt")
            lbl_path = f"{xview_lbl_dir}/{lbl_name}"
            if os.path.exists(lbl_path):
                os.symlink(lbl_path, f"{BASE}/labels/{split}/{lbl_name}")

link_xview_split(train_sources, "train")
link_xview_split(val_sources, "val")

# DOTA train
link_folder(f"{KAGGLE_INPUT}/kefanzhan/dotav1/train/images", f"{BASE}/images/train")
link_folder(f"{KAGGLE_INPUT}/henocamavigan/aerial-unified-22classes/dota_train_labels_unified", f"{BASE}/labels/train")

# DOTA val
link_folder(f"{KAGGLE_INPUT}/kefanzhan/dotav1/val/images", f"{BASE}/images/val")
link_folder(f"{KAGGLE_INPUT}/henocamavigan/aerial-unified-22classes/dota_val_labels_unified", f"{BASE}/labels/val")

print("images/train:", len(os.listdir(f"{BASE}/images/train")))
print("labels/train:", len(os.listdir(f"{BASE}/labels/train")))
print("images/val:", len(os.listdir(f"{BASE}/images/val")))
print("labels/val:", len(os.listdir(f"{BASE}/labels/val")))

images/train: 38018
labels/train: 38018
images/val: 7834
labels/val: 7834


## 3. Taxonomie unifiée (22 classes) et contrôle de la distribution

In [3]:
UNIFIED_CLASSES = [
    # 6 communes
    "plane", "helicopter", "ship", "storage-tank", "small-vehicle", "large-vehicle",
    # 9 DOTA-only
    "harbor", "bridge", "tennis-court", "swimming-pool", "basketball-court",
    "soccer-ball-field", "ground-track-field", "baseball-diamond", "roundabout",
    # 7 xView-only
    "building", "vehicle-lot", "railway-vehicle", "shipping-container",
    "engineering-vehicle", "construction-site", "infrastructure-diverse",
]
UNIFIED_NAME_TO_ID = {name: i for i, name in enumerate(UNIFIED_CLASSES)}
print(len(UNIFIED_CLASSES), "classes unifiées")
print(UNIFIED_NAME_TO_ID)

22 classes unifiées
{'plane': 0, 'helicopter': 1, 'ship': 2, 'storage-tank': 3, 'small-vehicle': 4, 'large-vehicle': 5, 'harbor': 6, 'bridge': 7, 'tennis-court': 8, 'swimming-pool': 9, 'basketball-court': 10, 'soccer-ball-field': 11, 'ground-track-field': 12, 'baseball-diamond': 13, 'roundabout': 14, 'building': 15, 'vehicle-lot': 16, 'railway-vehicle': 17, 'shipping-container': 18, 'engineering-vehicle': 19, 'construction-site': 20, 'infrastructure-diverse': 21}


In [4]:
from collections import Counter
import os

def count_classes(lbl_dir):
    dist = Counter()
    for fname in os.listdir(lbl_dir):
        path = f"{lbl_dir}/{fname}"
        if os.path.getsize(path) == 0:
            continue
        with open(path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cid = int(parts[0])
                dist[UNIFIED_CLASSES[cid]] += 1
    return dist

dist_train = count_classes(f"{BASE}/labels/train")
dist_val = count_classes(f"{BASE}/labels/val")

print("=== TRAIN ===")
for name, count in sorted(dist_train.items(), key=lambda x: -x[1]):
    print(f"{name:25s} {count}")

print("\n=== VAL ===")
for name, count in sorted(dist_val.items(), key=lambda x: -x[1]):
    print(f"{name:25s} {count}")

print("\nClasses absentes en train:", set(UNIFIED_CLASSES) - set(dist_train.keys()))
print("Classes absentes en val:", set(UNIFIED_CLASSES) - set(dist_val.keys()))

=== TRAIN ===
building                  327638
small-vehicle             253258
large-vehicle             74267
ship                      67908
plane                     18761
harbor                    13852
storage-tank              11321
vehicle-lot               5115
tennis-court              4885
engineering-vehicle       4635
shipping-container        4339
railway-vehicle           3816
bridge                    3727
infrastructure-diverse    3529
swimming-pool             3409
construction-site         1518
helicopter                1301
basketball-court          1170
soccer-ball-field         956
ground-track-field        910
baseball-diamond          873
roundabout                801

=== VAL ===
building                  35778
small-vehicle             31365
ship                      20313
large-vehicle             12928
plane                     5852
storage-tank              5289
harbor                    5039
tennis-court              1633
bridge                    924
swim

In [5]:
import yaml

data_yaml_fusionne = {
    "path": BASE,
    "train": "images/train",
    "val": "images/val",
    "nc": len(UNIFIED_CLASSES),
    "names": UNIFIED_CLASSES,
}

chemin_yaml_fusion = f"{BASE}/data_merged.yaml"
with open(chemin_yaml_fusion, "w") as f:
    yaml.dump(data_yaml_fusionne, f, sort_keys=False)

print("nc :", data_yaml_fusionne["nc"])

nc : 22


## 4. Sauvegarde des points de contrôle sur Kaggle

Une session Kaggle dure au plus 12 h. À chaque époque, le dossier du run est copié et, toutes les 5 époques, publié comme nouvelle version d'un dataset Kaggle : l'entraînement reprend ensuite avec `resume=True`. Nécessite des identifiants Kaggle configurés dans la session (jamais écrits dans le notebook).

In [6]:
import os, json

CKPT_BACKUP_DIR = f"{WORK_DIR}/ckpt_backup"
os.makedirs(CKPT_BACKUP_DIR, exist_ok=True)

os.system(f"kaggle datasets init -p '{CKPT_BACKUP_DIR}'")

meta_path = f"{CKPT_BACKUP_DIR}/dataset-metadata.json"
with open(meta_path) as f:
    meta = json.load(f)
meta["id"] = CKPT_DATASET_ID
meta["title"] = "xView DOTA Checkpoint Backup"
with open(meta_path, "w") as f:
    json.dump(meta, f)

print("Metadata recréée:", os.path.exists(meta_path))

Data package template written to: /kaggle/working/ckpt_backup/dataset-metadata.json
Metadata recréée: True


In [12]:
import json

with open(f"{CKPT_BACKUP_DIR}/split_sources.json", "w") as f:
    json.dump({"val": sorted(val_sources), "train": sorted(train_sources)}, f)

In [7]:
import shutil

def backup_checkpoint(trainer):
    if int(os.environ.get("RANK", "-1")) not in (-1, 0):
        return

    run_src = str(trainer.save_dir)
    run_dest = f"{CKPT_BACKUP_DIR}/run_state"
    shutil.copytree(run_src, run_dest, dirs_exist_ok=True)

    epoch = int(trainer.epoch) + 1
    print(f"[backup local] epoch {epoch} sauvegardé")

    if epoch % 5 == 0:
        code = os.system(
            f"kaggle datasets version -p '{CKPT_BACKUP_DIR}' "
            f"-m 'checkpoint epoch {epoch}' -r zip"
        )
        print(f"[backup Kaggle] epoch {epoch} | code retour: {code}")

print("Fonction backup_checkpoint redéfinie.")

Fonction backup_checkpoint redéfinie.


In [13]:
os.system(f"kaggle datasets status {CKPT_DATASET_ID}")

ready

0

## 5. Entraînement YOLOv8s (ou reprise)

In [9]:
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
import shutil, os

CKPT_BACKUP_DIR = f"{WORK_DIR}/ckpt_backup"
os.makedirs(CKPT_BACKUP_DIR, exist_ok=True)

last_ckpt = f"{CKPT_BACKUP_DIR}/run_state/weights/last.pt"

# Récupère le backup depuis le dataset Kaggle attaché en input
backup_input_path = f"{KAGGLE_INPUT}/ayiamavigan/xview-dota-ckpt-backup/weights/last.pt"
if os.path.exists(backup_input_path):
    os.makedirs(os.path.dirname(last_ckpt), exist_ok=True)
    shutil.copytree(
        os.path.dirname(os.path.dirname(backup_input_path)),
        f"{CKPT_BACKUP_DIR}/run_state",
        dirs_exist_ok=True,
    )

if os.path.exists(last_ckpt):
    print("Run existant détecté, reprise depuis:", last_ckpt)
    model = YOLO(last_ckpt)
    model.add_callback("on_train_epoch_end", backup_checkpoint)
    results = model.train(resume=True, device=[0, 1], save_period=-1)
else:
    print("Aucun run existant, démarrage à partir de zéro avec yolov8s.pt")
    model = YOLO("yolov8s.pt")
    model.add_callback("on_train_epoch_end", backup_checkpoint)
    results = model.train(
        data=chemin_yaml_fusion,
        epochs=150,
        imgsz=640,
        batch=32,
        device=[0, 1],
        patience=25,
        save_period=-1,
        cache="disk",
        project=f"{WORK_DIR}/runs",
        name="xview_dota_unified_v1",
    )

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Run existant détecté, reprise depuis: /kaggle/working/ckpt_backup/run_state/weights/last.pt
Ultralytics 8.4.163 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/train_data/data_merged.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.

100%|██████████| 6.27k/6.27k [00:00<00:00, 26.5kB/s]


Upload successful: split_sources.json (6KB)
Starting upload for file run_state.zip


100%|██████████| 119M/119M [00:01<00:00, 88.8MB/s]


Upload successful: run_state.zip (119MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/ayiamavigan/xview-dota-ckpt-backup
[backup Kaggle] epoch 60 | code retour: 0
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 123/123 1.1it/s 1:480.4ss
                   all       7834     125142      0.698      0.553      0.573       0.36


/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     61/150      6.37G      1.572      1.018      1.099         55        640: 100% ━━━━━━━━━━━━ 1189/1189 2.5it/s 7:550.2ss
[backup local] epoch 61 sauvegardé
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 123/123 1.2it/s 1:430.8ss
                   all       7834     125142      0.697      0.554      0.574       0.36

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     62/150      6.38G      1.561      1.017      1.097         23        640: 100% ━━━━━━━━━━━━ 1189/1189 2.4it/s 8:100.3ss
[backup local] epoch 62 sauvegardé
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 123/123 1.2it/s 1:460.4ss
                   all       7834     125142      0.699      0.555      0.573       0.36

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Si

## 6. Évaluation de `best.pt` sur la validation, par classe

In [11]:
from ultralytics import YOLO

model = YOLO(f"{KAGGLE_INPUT}/ayiamavigan/xview-dota-ckpt-backup/run_state/weights/best.pt")
metrics = model.val(data=f"{WORK_DIR}/train_data/data_merged.yaml", split="val")

for i, name in model.names.items():
    print(f"{name:25s} | AP50: {metrics.box.ap50[i]:.3f} | AP50-95: {metrics.box.ap[i]:.3f}")

Ultralytics 8.4.163 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 72 layers, 11,134,098 parameters, 0 gradients, 28.5 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.6±0.6 ms, read: 40.4±17.5 MB/s, size: 1345.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/working/train_data/labels/val... 7834 images, 3060 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 7834/7834 135.1it/s 58.0s.0sss
val: New cache created: /kaggle/working/train_data/labels/val.cache
WARNING ⚠️ Dataset images contain up to 705 objects (val=705), but max_det=300. This mismatch can cap recall and produce invalid validation metrics. Raising it may increase validation cost but cannot increase model or export capacity, which may cap recall. Setting max_det=705 to match the observed maximum.
                 Class     Images  Instances      Box(P          R      mA